# Practice #4. "ARIMA models and advanced techniques"

This notebook is dedicated to:
* Predicting Time series: Moving Average Model
* ARMA and ARIMA Models
* Model Selection and Diagnostics
* Stationarity Testing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import kpss
from scipy.stats import boxcox
import itertools

import warnings
warnings.filterwarnings('ignore')

## 0. Data reading and visualization

Please, specify path to data

In [ ]:
path_to_datafile = "../data/airline-passengers.csv"

In [ ]:
# data reading to pandas.DataFrame
df = pd.read_csv(path_to_datafile)

Please, rename time column to `ds` and data column to `y`(you can use `df.rename`) . If use dataset with multiple features select only one and drop NaN values

In [ ]:
# your code here
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)

Convert date column to datetime format and set as index

In [ ]:
df["ds"] = pd.to_datetime(df["ds"])
df.set_index("ds", inplace=True)

Number of data data points:

In [ ]:
df.shape[0]

Print slice of the timeseries:

In [ ]:
df.head()

Let's plot the data

In [ ]:
plt.figure(figsize=(20, 5))
plt.ylabel("y")
plt.xlabel("ds")
plt.plot(df);

## 1. Predicting Time series: Moving Average Model

The residual errors from forecasts on a time series provide another source of information that we can model. Residual errors themselves from a time series that can have a temporal structure. A simple autoregression model of this structure can be used to predict the forecast error, which in turn can be used to correct forecasts. This type of model is called a moving average model, the same name but very different from moving average smoothing. The difference between what was ground truth and what was predicted is called the residual error.<br>
Just like the input observations themselves, the residual errors from a time series can have a temporal structure like trends, bias, and seasonality. Any temporal structure in the time series of residual forecast errors is useful as a diagnostic as it suggests information that could be incorporated into the predictive model. An ideal model would leave no structure in the residual error, just random fluctuations that cannot be modeled.
Structure in the residual error can also be modeled directly. There may be complex signals in the residual error that are difficult to directly incorporate into the model. Instead, you can create a model of the residual error time series and predict the expected error for your model. The predicted error can then be subtracted from the model prediction and in turn, provide an additional lift in performance.

Split data on train and test set.

In [ ]:
# your code here
# df_train = ...
# df_test = ...

Please, calculate residual error using AR model and train set.

In [ ]:
#your code here
# df_forecast = ...

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(df_train, label='train')
plt.plot(df_forecast, label='forecast')
plt.ylabel("y")
plt.xlabel("ds")
plt.legend();

Please, train AR model for residual error

In [ ]:
# your code here
# re_forecast = ...

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(residual_error, label='residual error')
plt.plot(re_forecast, label='modeled residual error')
plt.ylabel("y")
plt.xlabel("ds")
plt.legend();

Now correct predictions with a AR model of residuals. With a good estimate of forecast error at a time step, we can make better predictions. For example, we can add the expected forecast error to a prediction to correct it and in turn improve the skill of the model.
$$\textit{improved forecast = forecast + estimated error}$$

In [ ]:
# your code here
# corrected_forecast = ...

In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(df_train, label='train')
plt.plot(df_test, label='test')
plt.plot(corrected_forecast, label='corrected forecast')
plt.ylabel("y")
plt.xlabel("ds")
plt.legend();

Moving Average model RMSE:

In [ ]:
np.sqrt(mean_squared_error(df_test['y'], corrected_forecast))

## 2. ARMA and ARIMA Models

ARIMA (AutoRegressive Integrated Moving Average) models are a powerful class of models for analyzing and forecasting time series data. They combine three components:

- **AR (AutoRegressive)**: Uses past values to predict future values
- **I (Integrated)**: Uses differencing to make the series stationary
- **MA (Moving Average)**: Uses past forecast errors to predict future values

ARIMA(p,d,q) where:
- p: order of autoregression (AR)
- d: degree of differencing (I)
- q: order of moving average (MA)

### Mathematical Representation:

**AR(p) model:**
$$y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + ... + \phi_p y_{t-p} + \epsilon_t$$

**MA(q) model:**
$$y_t = c + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + ... + \theta_q \epsilon_{t-q}$$

**ARIMA(p,d,q) model:**
$$(1 - \phi_1 L - \phi_2 L^2 - ... - \phi_p L^p)(1-L)^d y_t = (1 + \theta_1 L + \theta_2 L^2 + ... + \theta_q L^q)\epsilon_t$$

Where L is the lag operator.

### 2.1 Stationarity Testing and Data Preparation

Before fitting ARIMA models, we need to ensure our time series is stationary. Let's test for stationarity and determine the order of differencing (d).

In [ ]:
def check_stationarity(timeseries, title):
    """
    Check stationarity using ADF and KPSS tests
    """
    print(f'Results of Stationarity Tests for {title}:')
    print('-' * 50)
    
    # Augmented Dickey-Fuller test
    adf_result = adfuller(timeseries.dropna())
    print('ADF Statistic:', adf_result[0])
    print('p-value:', adf_result[1])
    print('Critical Values:')
    for key, value in adf_result[4].items():
        print(f'\t{key}: {value}')
    
    if adf_result[1] <= 0.05:
        print("ADF Test: Series is stationary")
    else:
        print("ADF Test: Series is non-stationary")
    
    # KPSS test
    kpss_result = kpss(timeseries.dropna())
    print(f'\nKPSS Statistic: {kpss_result[0]}')
    print(f'p-value: {kpss_result[1]}')
    print('Critical Values:')
    for key, value in kpss_result[3].items():
        print(f'\t{key}: {value}')
    
    if kpss_result[1] >= 0.05:
        print("KPSS Test: Series is stationary")
    else:
        print("KPSS Test: Series is non-stationary")
    
    print('\n')

# Check stationarity of original series
check_stationarity(df['y'], 'Original Series')

Apply differencing to make the series stationary if needed:

In [ ]:
# your code here
# Apply differencing
# ...


### 2.2 Parameter Selection (p, d, q)

Use ACF and PACF plots to identify appropriate p and q parameters:

In [ ]:
# your code here
# Plot ACF and PACF for stationary series
# ...

# Guidelines for parameter selection:
# - PACF cuts off after lag p - AR(p) model
# - ACF cuts off after lag q - MA(q) model
# - Both trail off - ARMA(p,q) model

### 2.3 Automatic Model Selection

Implement grid search to find optimal ARIMA parameters using information criteria:

In [ ]:
def find_best_arima(ts, max_p=3, max_d=2, max_q=3):
    """
    Find best ARIMA parameters using grid search and AIC
    """
    best_aic = np.inf
    best_params = None
    best_model = None
    
    results = []
    
    # Generate all combinations of p, d, q
    for p in range(max_p + 1):
        for d in range(max_d + 1):
            for q in range(max_q + 1):
                try:
                    model = ARIMA(ts, order=(p, d, q))
                    fitted_model = model.fit()
                    
                    aic = fitted_model.aic
                    bic = fitted_model.bic
                    
                    results.append({
                        'p': p, 'd': d, 'q': q,
                        'AIC': aic, 'BIC': bic
                    })
                    
                    if aic < best_aic:
                        best_aic = aic
                        best_params = (p, d, q)
                        best_model = fitted_model
                        
                except Exception as e:
                    continue
    
    results_df = pd.DataFrame(results)
    print("Top 10 models by AIC:")
    print(results_df.sort_values('AIC').head(10))
    
    print(f"\nBest ARIMA parameters: {best_params}")
    print(f"Best AIC: {best_aic:.2f}")
    
    return best_model, best_params, results_df

# your code here
# Split data for training and testing
# ...

# Find best ARIMA model
# ...

# Print model summary
# ...

### 2.4 ARIMA Forecasting

Make forecasts using the best ARIMA model:

In [ ]:
# your code here
# Make forecasts
# ...

# Create forecast index
# ...

# Plot results
# ...

# Calculate performance metrics
# ...

### 2.5 Model Diagnostics

Perform residual analysis to validate the model:

In [ ]:
# your code here
# Residual analysis
# ...

# # Residuals plot
# ...

# # Histogram of residuals
# ...

# # Q-Q plot
# ...

# # ACF of residuals
# ...

# # Ljung-Box test for autocorrelation in residuals
# ...